In [21]:
"""
RunSignUp API - Race results for a given race and result set.
Docs: https://runsignup.com/Api/race/:race_id/results/get-results/GET
"""

import requests

API_KEY    = "TkPIUG2GLK95zjIAONwAyv348fQQ6TtY"
API_SECRET = "oNsju6QeN0UEVZ1JK612wyay0dC2jLJ9"
AFLT_TOKEN = "uL3LJsNhaU6SFxIRWfJFjA0K24o2fddK"

DB_PATH = r"C:\Users\Ben Sylve\Documents\git_sf\sf-pipeline\db\race_analytics.duckdb"

In [22]:
import duckdb

con = duckdb.connect(DB_PATH, read_only=True)
rows = con.execute("""
    SELECT
        race_event_id,
        race_event_name,
        race_event_data_source_id,
        race_event_metadata_1_key, race_event_metadata_1_value,
        race_event_metadata_2_key, race_event_metadata_2_value,
        race_event_metadata_3_key, race_event_metadata_3_value
    FROM silver.race_event
    ORDER BY race_date
""").fetchall()
con.close()

events = []
for row in rows:
    meta = {k: v for k, v in [(row[3], row[4]), (row[5], row[6]), (row[7], row[8])] if k is not None}
    result_set_key = next((k for k in meta if k not in ("RACE_ID", "EVENT_ID")), None)
    events.append({
        "race_event_id":             row[0],
        "race_event_name":           row[1],
        "race_event_data_source_id": row[2],
        "race_id":                   int(meta["RACE_ID"]),
        "event_id":                  int(meta["EVENT_ID"]),
        "result_set_id":             int(meta[result_set_key]) if result_set_key else None,
    })

print(f"{len(events)} event(s) found")
for e in events:
    print(f"  {e['race_event_id']}: {e['race_event_name']}")

1 event(s) found
  WC_2025: Watermelon Classic 2025


In [23]:
def get_all_results(race_id, event_id, result_set_id=None):
    url = f"https://runsignup.com/Rest/race/{race_id}/results/get-results"
    all_results = []
    page = 1

    while True:
        params = {
            "api_key":    API_KEY,
            "api_secret": API_SECRET,
            "aflt_token": AFLT_TOKEN,
            "format":     "json",
            "event_id":   event_id,
            "page":       page,
            "results_per_page": 250,
        }
        if result_set_id:
            params["individual_result_set_id"] = result_set_id

        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()

        page_results = []
        for rs in data.get("individual_results_sets", []):
            page_results.extend(rs.get("results", []))

        if not page_results:
            break

        all_results.extend(page_results)
        print(f"Page {page}: {len(page_results)} results")
        page += 1

    return all_results

In [24]:
import pandas as pd

for event in events:
    print(f"\n--- {event['race_event_name']} ({event['race_event_id']}) ---")
    results = get_all_results(event["race_id"], event["event_id"], result_set_id=event["result_set_id"])
    print(f"Total: {len(results)} results")

    df = pd.DataFrame(results)
    df = df[[c for c in df.columns if not c.startswith("division-")]]
    df["race_event_id"]             = event["race_event_id"]
    df["race_event_data_source_id"] = event["race_event_data_source_id"]

    table = f"bronze.race_results_{event['race_event_data_source_id']}__{event['race_event_id']}"

    con = duckdb.connect(DB_PATH)
    con.execute("CREATE SCHEMA IF NOT EXISTS bronze")
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM df")
    con.close()

    print(f"Saved to {table}")
    print(df.head())


--- Watermelon Classic 2025 (WC_2025) ---
Page 1: 250 results
Page 2: 250 results
Page 3: 250 results
Page 4: 230 results
Total: 980 results
Saved to bronze.race_results_RUNSIGNUP__WC_2025
   result_id  place  bib      first_name last_name gender  city state  \
0  188309290      1  880          Speed,   Preston      M  None  None   
1  188309291      2  405         Porter,      Will      M  None  None   
2  188309292      3  337     Richardson,    Slater      M  None  None   
3  188309293      4  460  Ward-McCammon,   Dominic      M  None  None   
4  188309294      5  347          Gomez,      Adam      M  None  None   

  country_code clock_time chip_time  pace  age age_percentage race_event_id  \
0         None      16:21     16:21  5:15   22          78.6%       WC_2025   
1         None      16:46     16:45  5:23   22          76.7%       WC_2025   
2         None      16:54     16:53  5:25   25          76.1%       WC_2025   
3         None      17:03     17:02  5:28   22         